# 第三部分：TensorRT 部署详解（使用者指南）

TensorRT 是 NVIDIA 的 **GPU 推理加速库**。  
对使用者来说，核心就三件事：  
**1. 把 ONNX 模型构建成 TensorRT Engine**  
**2. 加载 Engine 进行推理（Python / C++）**  
**3. 按需使用 FP16/INT8 等优化**

---

## 一、TensorRT 是什么（最简概念）

- 作用：将训练好的模型针对特定 NVIDIA GPU **编译优化**，达到极致推理速度
- 本质：**编译器 + 运行时**
- 核心优化：算子融合、低精度（FP16/INT8）、内存复用
- 输入必须是 ONNX（或通过其他方式转换），不能直接加载 `.pth`

完整工作流：
```text
PyTorch → ONNX → TensorRT Engine → 超快推理
```

---

## 二、整体流程

```text
第一步：导出 ONNX（前面已讲）
第二步：构建 Engine（一次构建，重复加载）
第三步：加载 Engine 推理（Python / C++）
```

---

## 三、构建 Engine（trtexec 工具）

TensorRT 自带命令行工具 `trtexec`，无需写代码即可构建和测试 Engine。

### 3.1 安装 TensorRT
- 从 [NVIDIA Developer](https://developer.nvidia.com/tensorrt) 下载对应 CUDA 版本的 TensorRT（tar 包或 deb 包）
- 安装 Python 绑定：`pip install tensorrt`（需与系统 TensorRT 版本一致）
- 安装 pycuda：`pip install pycuda`（Python 推理时需要）

### 3.2 常用 trtexec 命令

| 需求 | 命令 | 关键参数说明 |
|------|------|-------------|
| 构建 FP32 Engine | `trtexec --onnx=model.onnx --saveEngine=model.engine` | `--onnx` 输入模型，`--saveEngine` 输出路径 |
| 构建 FP16 Engine | `trtexec --onnx=model.onnx --fp16 --saveEngine=model_fp16.engine` | `--fp16` 开启半精度，速度 1.5~3× 提升 |
| 构建 INT8 Engine | `trtexec --onnx=model.onnx --int8 --saveEngine=model_int8.engine --calib=calib.cache` | 需要校准数据，加速 3~8× |
| 动态 Shape Engine | `trtexec --onnx=model.onnx --minShapes=input:1x3x224x224 --optShapes=input:8x3x224x224 --maxShapes=input:32x3x224x224 --saveEngine=model_dyn.engine` | `minShapes` 最小，`optShapes` 最优（实际推理时越接近越好），`maxShapes` 最大 |
| 直接测速 | `trtexec --loadEngine=model.engine --shapes=input:8x3x224x224 --iterations=1000` | `--shapes` 指定测试形状，`--iterations` 推理次数 |

**常用可选参数**：
- `--workspace=2048`：限制最大工作显存（MiB），构建时需要

**动态 Shape 提示**：如果 ONNX 导出时用了 `dynamic_axes`，构建 Engine 必须指定 `minShapes / optShapes / maxShapes`，否则会报错。

---

## 四、TensorRT Python 推理核心 API

### 4.1 加载 Engine
```python
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit
import numpy as np

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
runtime = trt.Runtime(TRT_LOGGER)

with open("model.engine", "rb") as f:
    engine = runtime.deserialize_cuda_engine(f.read())
```

### 4.2 创建执行上下文
```python
context = engine.create_execution_context()
```
上下文负责管理输入/输出绑定和执行。

### 4.3 分配 GPU 内存并准备输入（简单示例）
假设模型有一个输入和一个输出，且形状固定：
```python
# 输入输出形状
input_shape = (1, 3, 224, 224)
output_shape = (1, 1000)

# 输入数据（numpy，float32）
input_data = np.random.randn(*input_shape).astype(np.float32)

# 分配 GPU 显存
d_input = cuda.mem_alloc(input_data.nbytes)
d_output = cuda.mem_alloc(np.prod(output_shape) * 4)  # 4 bytes per float32

# 拷贝数据到 GPU
cuda.memcpy_htod(d_input, input_data)

# 绑定输入输出
bindings = [int(d_input), int(d_output)]
```

### 4.4 执行推理
```python
context.execute_v2(bindings)
```

### 4.5 取回输出
```python
output = np.empty(output_shape, dtype=np.float32)
cuda.memcpy_dtoh(output, d_output)
```

### 4.6 动态 Shape 推理
如果 Engine 支持动态 shape，需要先设置输入形状：
```python
context.set_input_shape("input", input_data.shape)  # "input" 必须与 ONNX 的 input_names 一致
```

### 4.7 完整推理函数（封装版）
```python
def infer(engine_path, input_np):
    logger = trt.Logger(trt.Logger.WARNING)
    runtime = trt.Runtime(logger)
    with open(engine_path, "rb") as f:
        engine = runtime.deserialize_cuda_engine(f.read())
    context = engine.create_execution_context()

    # 设置动态 shape（如果需要）
    context.set_input_shape("input", input_np.shape)

    # 分配显存
    d_input = cuda.mem_alloc(input_np.nbytes)
    output_shape = context.get_tensor_shape("output")  # 获取实际输出形状
    d_output = cuda.mem_alloc(np.prod(output_shape) * 4)
    bindings = [int(d_input), int(d_output)]

    # 拷贝输入
    cuda.memcpy_htod(d_input, input_np)
    context.execute_v2(bindings)

    # 拷贝输出
    output_np = np.empty(output_shape, dtype=np.float32)
    cuda.memcpy_dtoh(output_np, d_output)
    return output_np
```

**核心 API 速记**：
- `trt.Runtime(logger)` → 运行时
- `runtime.deserialize_cuda_engine(engine_bytes)` → 加载 engine
- `engine.create_execution_context()` → 创建上下文
- `context.set_input_shape(name, shape)` → 动态 shape 设置
- `context.execute_v2(bindings)` → 执行推理
- `cuda.mem_alloc` / `cuda.memcpy_htod` / `cuda.memcpy_dtoh` → GPU 内存管理

---

## 五、TensorRT C++ 推理环境配置

### 5.1 安装
TensorRT 包中已包含 C++ 库。目录结构：
```text
TensorRT-8.x.x.x/
├── include/          # 头文件：NvInfer.h, NvInferRuntime.h 等
└── lib/              # 动态库：libnvinfer.so, libnvinfer_plugin.so 等
```
还需要 CUDA Toolkit（默认应该已安装）。

### 5.2 CMakeLists.txt 示例
```cmake
cmake_minimum_required(VERSION 3.18)
project(TRTDemo)

set(CMAKE_CXX_STANDARD 17)

# TensorRT 路径
set(TRT_ROOT "/path/to/TensorRT-8.x.x.x")

include_directories(${TRT_ROOT}/include)
link_directories(${TRT_ROOT}/lib)

add_executable(main main.cpp)
target_link_libraries(main
    nvinfer nvinfer_plugin        # TensorRT 核心库
    cuda cudart                  # CUDA 运行时
)
```

### 5.3 编译
```bash
mkdir build && cd build
cmake ..
make
```

---

## 六、TensorRT C++ 最常用 API

### 6.1 加载 Engine
```cpp
#include <NvInfer.h>
#include <fstream>
#include <iostream>
#include <vector>

class Logger : public nvinfer1::ILogger {
    void log(Severity severity, const char* msg) noexcept override {
        if (severity != Severity::kINFO) std::cerr << msg << std::endl;
    }
};

Logger logger;
nvinfer1::IRuntime* runtime = nvinfer1::createInferRuntime(logger);

std::ifstream engine_file("model.engine", std::ios::binary);
engine_file.seekg(0, std::ios::end);
size_t size = engine_file.tellg();
engine_file.seekg(0, std::ios::beg);
std::vector<char> engine_data(size);
engine_file.read(engine_data.data(), size);

nvinfer1::ICudaEngine* engine = runtime->deserializeCudaEngine(engine_data.data(), size);
```

### 6.2 创建执行上下文
```cpp
nvinfer1::IExecutionContext* context = engine->createExecutionContext();
```

### 6.3 准备输入输出（固定 shape）
```cpp
// 假设输入名为 "input"，输出名为 "output"（与 ONNX 导出时一致）
const char* input_name = "input";
const char* output_name = "output";

// 获取输入输出维度
nvinfer1::Dims input_dims = engine->getBindingDimensions(0);  // 第 0 个绑定是输入
nvinfer1::Dims output_dims = engine->getBindingDimensions(1); // 第 1 个绑定是输出

// 计算元素个数
int input_size = 1;
for (int i = 0; i < input_dims.nbDims; i++) input_size *= input_dims.d[i];
int output_size = 1;
for (int i = 0; i < output_dims.nbDims; i++) output_size *= output_dims.d[i];

// 分配内存（host + device）
float* h_input = new float[input_size];
float* h_output = new float[output_size];

void* d_input, * d_output;
cudaMalloc(&d_input, input_size * sizeof(float));
cudaMalloc(&d_output, output_size * sizeof(float));

// 绑定
void* bindings[] = {d_input, d_output};
```

### 6.4 拷贝输入并推理
```cpp
// 填充数据... h_input
cudaMemcpy(d_input, h_input, input_size * sizeof(float), cudaMemcpyHostToDevice);

// 执行
context->executeV2(bindings);

// 取回输出
cudaMemcpy(h_output, d_output, output_size * sizeof(float), cudaMemcpyDeviceToHost);
```

### 6.5 动态 Shape 设置
```cpp
context->setBindingDimensions(0, nvinfer1::Dims4{1, 3, 224, 224});  // 先设置输入 shape
// 然后才能分配正确大小的输出内存
```

### 6.6 释放资源
```cpp
delete[] h_input;
delete[] h_output;
cudaFree(d_input);
cudaFree(d_output);
context->destroy();
engine->destroy();
runtime->destroy();
```

### 6.7 C++ 最小可运行示例（完整）
```cpp
#include <NvInfer.h>
#include <cuda_runtime_api.h>
#include <fstream>
#include <iostream>
#include <vector>

class Logger : public nvinfer1::ILogger {
    void log(Severity severity, const char* msg) noexcept override {}
} logger;

int main() {
    // 加载 engine
    std::ifstream f("model.engine", std::ios::binary);
    std::vector<char> data((std::istreambuf_iterator<char>(f)), std::istreambuf_iterator<char>());
    nvinfer1::IRuntime* runtime = nvinfer1::createInferRuntime(logger);
    nvinfer1::ICudaEngine* engine = runtime->deserializeCudaEngine(data.data(), data.size());
    nvinfer1::IExecutionContext* context = engine->createExecutionContext();

    // 获取输入输出索引
    int input_idx = engine->getBindingIndex("input");
    int output_idx = engine->getBindingIndex("output");

    // 获取形状
    auto input_dims = engine->getBindingDimensions(input_idx);
    auto output_dims = engine->getBindingDimensions(output_idx);
    int input_size = 1, output_size = 1;
    for (int i = 0; i < input_dims.nbDims; i++) input_size *= input_dims.d[i];
    for (int i = 0; i < output_dims.nbDims; i++) output_size *= output_dims.d[i];

    // 分配内存
    float *h_in = new float[input_size], *h_out = new float[output_size];
    void *d_in, *d_out;
    cudaMalloc(&d_in, input_size * sizeof(float));
    cudaMalloc(&d_out, output_size * sizeof(float));

    // 推理
    cudaMemcpy(d_in, h_in, input_size * sizeof(float), cudaMemcpyHostToDevice);
    void* bindings[] = {d_in, d_out};
    context->executeV2(bindings);
    cudaMemcpy(h_out, d_out, output_size * sizeof(float), cudaMemcpyDeviceToHost);

    // 清理
    delete[] h_in; delete[] h_out;
    cudaFree(d_in); cudaFree(d_out);
    context->destroy();
    engine->destroy();
    runtime->destroy();
    return 0;
}
```

---

## 七、常用 API 速查表

| 操作 | Python API | C++ API |
|------|-----------|--------|
| 创建运行时 | `trt.Runtime(logger)` | `nvinfer1::createInferRuntime(logger)` |
| 加载引擎 | `runtime.deserialize_cuda_engine(bytes)` | `runtime->deserializeCudaEngine(data, size)` |
| 创建上下文 | `engine.create_execution_context()` | `engine->createExecutionContext()` |
| 设置动态形状 | `context.set_input_shape(name, shape)` | `context->setBindingDimensions(idx, dims)` |
| 执行推理 | `context.execute_v2(bindings)` | `context->executeV2(bindings)` |
| 获取绑定维度 | `engine.get_tensor_shape(name)` | `engine->getBindingDimensions(idx)` |
| GPU 分配 | `cuda.mem_alloc(size)` | `cudaMalloc(&ptr, size)` |
| 拷贝到 GPU | `cuda.memcpy_htod(dst, src)` | `cudaMemcpy(dst, src, size, cudaMemcpyHostToDevice)` |
| 拷贝回 CPU | `cuda.memcpy_dtoh(dst, src)` | `cudaMemcpy(dst, src, size, cudaMemcpyDeviceToHost)` |

---

## 八、常见问题

| 现象 | 原因 | 解决 |
|------|------|------|
| `trtexec` 解析 ONNX 失败 | 有不支持的算子 | 升级 TensorRT 或修改模型，用 `onnxsim` 简化 |
| 构建动态 shape 时报错 | 未配置 min/opt/maxShapes | 补充这三个参数 |
| 推理时 shape 不匹配 | 实际输入超出 min/max 范围 | 调整输入大小或重新构建 Engine |
| FP16/INT8 精度丢失严重 | 部分层敏感或校准不充分 | 检查校准数据质量，使用混合精度 |
| C++ 编译找不到头文件 | 路径未配置 | 在 CMake 中正确设置 `include_directories` |

---

## 九、总结

使用者只需掌握：
1. **构建 Engine**：`trtexec --onnx=model.onnx --fp16 --saveEngine=model.engine`
2. **Python 推理**：加载 engine → 分配显存 → `execute_v2` → 取结果
3. **C++ 推理**：同样流程，注意手动管理 CUDA 内存
